# 02 · Model training & experiment tracking

Thin wrapper around `python -m heart.train`. All runs go to the local MLflow store in `../mlruns`; open it with `mlflow ui --backend-store-uri ./mlruns` from the repo root.

In [ ]:
import os

import mlflow
import pandas as pd
from IPython.display import Image, display

from heart import config, train

os.chdir(config.PROJECT_ROOT)  # so ./mlruns resolves to the repo root

In [ ]:
summary = train.main(["--n-iter", "30"])  # 3 models x 3 feature sets
summary

## Compare runs

In [ ]:
runs = mlflow.search_runs(experiment_names=[config.MLFLOW_EXPERIMENT],
                          filter_string="tags.mlflow.parentRunId IS NULL")
cols = ["tags.mlflow.runName", "tags.feature_set", "metrics.cv_roc_auc_mean",
        "metrics.cv_roc_auc_std", "metrics.oof_brier", "metrics.oof_recall_female",
        "metrics.oof_recall_male", "metrics.test_roc_auc", "metrics.oof_threshold",
        "metrics.cv_overfit_gap"]
runs[cols].sort_values("metrics.cv_roc_auc_mean", ascending=False)

In [ ]:
display(Image(filename=str(config.FIGURES_DIR / "model_comparison.png")))

## Feature-set ablation
CV ROC-AUC for every model × feature set. Selection prefers the simpler feature set unless the richer one wins by more than the tolerance (0.005 AUC).

_Observation: did the risk bands or the HR ratio help? for which model family?_

In [ ]:
display(Image(filename=str(config.FIGURES_DIR / "feature_set_ablation.png")))

## Calibration and fairness
Out-of-fold reliability curves (does a predicted 0.7 mean ~70%?) and recall for women vs men at the chosen threshold.

_Observation:_

In [ ]:
display(Image(filename=str(config.FIGURES_DIR / "calibration_comparison.png")))
selected = summary["selected"]
display(Image(filename=str(config.FIGURES_DIR / selected / "subgroup_by_sex.png")))

## Model selection rationale
_Explain which model was selected and why (CV AUC, variance across folds, overfitting gap, interpretability, recall at the chosen threshold)._

In [ ]:
pd.read_csv(config.REPORTS_DIR / "model_comparison.csv").round(3)